# 🌍 Currency Recognition — Colab Training (India · Turkey · USA)
Pipeline: **note localisation (OpenCV) → currency / non-currency detector (plain CNN) → denomination classifier (ResNet50)**.
Run top to bottom. **Runtime → Change runtime type → T4 GPU** first.

## 1. GPU check + dependencies

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__)
print('GPU available:', bool(gpus), gpus)
!pip install -q kagglehub scikit-learn matplotlib tensorflow-datasets

## 1b. Checkpoints on Google Drive (so a disconnect never costs you the training)
Training now saves its full state **after every epoch** (weights + optimizer + epoch counter + best model).
Colab's local disk is wiped when the runtime ends, so the checkpoints go to **Google Drive**.

**If the runtime dies or the network drops:** reconnect → run cells 1, 1b, 2, 3, 4, 5 again → run the training cell again. It prints `[checkpoint] ... resuming after epoch N` and carries on from there. Finished phases are skipped.
To deliberately start over, run `!python train.py --fresh` (or `train_detector.py --fresh`).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['CURRENCY_CKPT_DIR'] = '/content/drive/MyDrive/currency_ckpt'   # read by config.py in the training scripts
os.makedirs(os.environ['CURRENCY_CKPT_DIR'], exist_ok=True)
print('Checkpoints ->', os.environ['CURRENCY_CKPT_DIR'])

## 2. Upload the project zip (`currency_recognition.zip`)

In [ ]:
from pathlib import Path
import os, zipfile
from google.colab import files

up = files.upload()                                   # choose currency_recognition.zip
zipfile.ZipFile(next(iter(up))).extractall('/content')
PROJECT = Path('/content/currency_recognition')
os.chdir(PROJECT)
print('Working directory:', Path.cwd())
!ls

## 3. Currency dataset (INR · TRY · USD)
**Option A (default):** download from Kaggle and organise into `dataset/<CODE>_<VALUE>/` automatically.
If Kaggle asks for login: run `import kagglehub; kagglehub.login()` in a new cell, then re-run.

In [ ]:
!python prepare_data.py

**Option B:** use your own dataset zip from Google Drive instead (folders `INR_10`, `TRY_100`, `USD_20` ...). Set `USE_DRIVE = True`.

In [ ]:
USE_DRIVE = False
DRIVE_ZIP = '/content/drive/MyDrive/currency_dataset.zip'     # zip whose top level contains the class folders
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    zipfile.ZipFile(DRIVE_ZIP).extractall(PROJECT / 'dataset')

## 4. Negative (NOT-currency) data — required for the detector
Caltech101 gives faces / objects / backgrounds. **Add your own webcam negatives for the best result**
(on your laptop: `python capture_images.py negative/face`, `negative/background`, `negative/random_objects`, then zip the `negative` folder).

In [ ]:
!python prepare_negatives.py

In [ ]:
# Optional: upload a zip of YOUR negative images (the `negative` folder zipped, or any folder of images)
up2 = files.upload()          # cancel / skip if you have none
for name in up2:
    zipfile.ZipFile(name).extractall(PROJECT / 'negative')

## 5. Verify dataset + class distribution

In [ ]:
!python check_dataset.py

In [ ]:
import matplotlib.pyplot as plt, dataset_utils as du
cls = {k: len(v) for k, v in du.currency_classes(max_per_class=None).items()}
neg = {k: len(v) for k, v in du.negative_groups().items()}
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(cls.keys(), cls.values()); ax[0].set_title('Currency images per class'); ax[0].tick_params(axis='x', rotation=90)
ax[1].bar(neg.keys() or ['none'], neg.values() or [0]); ax[1].set_title('Negative images per group'); ax[1].tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

## 6. Train the currency / non-currency detector (stage B)
*Safe to interrupt: re-running this cell resumes from the last finished epoch (needs cell 1b).*

In [ ]:
!python train_detector.py

## 7. Train the ResNet50 denomination classifier (stage C)
*Safe to interrupt: re-running this cell resumes from the last finished epoch (needs cell 1b).*

In [ ]:
!python train.py

## 8. Evaluate both models

In [ ]:
from IPython.display import Image, display
for f in ['detector_report.txt', 'classification_report.txt', 'open_set_report.txt']:
    print('=' * 20, f); print(open(f'outputs/{f}').read())
for f in ['detector_eval.png', 'training_curves.png', 'confusion_matrix.png']:
    display(Image(f'outputs/{f}'))

Check **detector_report.txt**: false-accept rate for every negative group should be near 0%.
If a group (e.g. `face`) is high, add more of those images and re-train. If recall on notes is low, lower `DETECTOR_THRESHOLD` slightly in `config.py`.

## 9. Save + download the trained models

In [ ]:
!ls -la model
!zip -qr trained_models.zip model
files.download('trained_models.zip')
# keep a copy on Drive as well (Drive is already mounted in cell 1b)
!cp trained_models.zip "$CURRENCY_CKPT_DIR"/


## 10. (Optional) Try the full pipeline on an image here

In [ ]:
img = files.upload()
for name in img:
    !python predict.py "{name}"

### Next (on your laptop)
Unzip `trained_models.zip` into the project's `model/` folder so it contains:
`currency_resnet50.weights.h5`, `class_names.json`, `classifier_meta.json`, `currency_detector.weights.h5`, `detector_meta.json`.
Then `python webcam.py` or `python -m streamlit run app.py`.